# 01 — Exploratory Data Analysis
## Diabetes Risk Prediction · Clinical ML Portfolio

**Author:** Alejandro Zakzuk — Physician · AI Applied to Health  
**Dataset:** Pima Indians Diabetes Database (NIDDK)  
**Goal:** Understand the data before any modeling decision. Identify distributions, missing data patterns, clinical outliers, and target imbalance.

> *Clinical note:* EDA in a clinical ML project is not just statistical housekeeping. It is where we decide whether the data is fit for purpose — and document the assumptions we carry into modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# plotting config
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded OK')

## 1. Load & First Look

In [ ]:
df = pd.read_csv('../data/diabetes.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
df.describe().round(2)

## 2. Target Variable Distribution

Class imbalance has direct implications for model evaluation. Accuracy becomes an unreliable metric if one class dominates — a model that predicts 'no diabetes' for everyone would achieve ~65% accuracy here, which is meaningless clinically.

In [ ]:
counts = df['Outcome'].value_counts()
pct = df['Outcome'].value_counts(normalize=True) * 100

print('Target distribution:')
print(f'  No diabetes (0): {counts[0]} ({pct[0]:.1f}%)')
print(f'  Diabetes    (1): {counts[1]} ({pct[1]:.1f}%)')
print(f'\nImbalance ratio: {counts[0]/counts[1]:.2f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No diabetes', 'Diabetes'], counts.values, color=['#4C72B0', '#C44E52'], alpha=0.85, edgecolor='white')
for bar, val, p in zip(bars, counts.values, pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{val}\n({p:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, 580)
plt.tight_layout()
plt.show()

print('\n→ Mild imbalance (~1.87:1). AUC-ROC is appropriate as primary metric.')
print('→ class_weight="balanced" or stratified CV will be used in modeling.')

## 3. Zero-Value Analysis — The Hidden Missing Data Problem

Several features contain biologically implausible zeros. A BMI of 0, glucose of 0, or diastolic BP of 0 are physiologically impossible. These represent **missing data encoded as zeros**, not true measurements. This is one of the most common data quality issues in clinical datasets and must be addressed before modeling.

In [ ]:
# Features where zero is biologically implausible
zero_implausible = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Zero-value counts in clinically implausible features:')
print('-' * 55)
for col in zero_implausible:
    n_zeros = (df[col] == 0).sum()
    pct_zeros = n_zeros / len(df) * 100
    print(f'  {col:<30} {n_zeros:>3} zeros ({pct_zeros:.1f}%)')

print('\nClinical interpretation:')
print('  Glucose=0        → impossible; likely missing OGTT result')
print('  BloodPressure=0  → impossible; likely missing BP measurement')
print('  SkinThickness=0  → implausible; likely not measured')
print('  Insulin=0        → possible but suspicious at 48.7%; likely not measured')
print('  BMI=0            → impossible; likely missing anthropometric data')
print('\n→ Strategy: replace zeros with NaN, then impute (see notebook 02)')

## 4. Feature Distributions by Outcome

In [ ]:
features = ['Glucose', 'BMI', 'Age', 'DiabetesPedigreeFunction',
            'Pregnancies', 'BloodPressure', 'SkinThickness', 'Insulin']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    for outcome, color, label in [(0, '#4C72B0', 'No diabetes'), (1, '#C44E52', 'Diabetes')]:
        subset = df[df['Outcome'] == outcome][feat].replace(0, np.nan).dropna()
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Density')
    if i == 0:
        axes[i].legend(fontsize=8)

fig.suptitle('Feature Distributions by Diabetes Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  Glucose: clearest separation — consistent with its role as diagnostic criterion')
print('  BMI: diabetic group shifted right — aligns with T2D pathophysiology')
print('  Age: older patients more represented in diabetic group')
print('  DiabetesPedigreeFunction: overlapping but diabetic group slightly higher')
print('  Insulin: high zero rate distorts distribution significantly')

## 5. Correlation Analysis

In [ ]:
# Replace implausible zeros before computing correlations
df_clean = df.copy()
for col in zero_implausible:
    df_clean[col] = df_clean[col].replace(0, np.nan)

corr = df_clean.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 9})
ax.set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with outcome
outcome_corr = corr['Outcome'].drop('Outcome').sort_values(ascending=False)
print('Correlation with Outcome (descending):')
for feat, val in outcome_corr.items():
    bar = '█' * int(abs(val) * 20)
    print(f'  {feat:<28} {val:+.3f}  {bar}')

## 6. Statistical Tests by Feature

Mann-Whitney U test (non-parametric) to assess whether feature distributions differ significantly between diabetic and non-diabetic groups. Appropriate given non-normality of several features.

In [ ]:
from scipy.stats import mannwhitneyu

print(f'Mann-Whitney U Test — Feature Differences by Diabetes Status')
print(f'{"Feature":<30} {"Mean (No DM)":>12} {"Mean (DM)":>10} {"p-value":>10} {"Significant":>12}')
print('-' * 78)

for feat in features:
    g0 = df_clean[df_clean['Outcome'] == 0][feat].dropna()
    g1 = df_clean[df_clean['Outcome'] == 1][feat].dropna()
    stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '*** p<0.001' if p < 0.001 else ('** p<0.01' if p < 0.01 else ('* p<0.05' if p < 0.05 else 'ns'))
    print(f'{feat:<30} {g0.mean():>12.2f} {g1.mean():>10.2f} {p:>10.4f} {sig:>12}')

print('\n→ Glucose, BMI, Age, and DiabetesPedigreeFunction show strongest separation.')
print('→ BloodPressure shows statistical significance but modest effect size — interpret with caution.')

## 7. Outlier Detection — Clinical Plausibility Check

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
priority_feats = ['Glucose', 'BMI', 'BloodPressure', 'Insulin']

for ax, feat in zip(axes, priority_feats):
    data = df_clean[feat].dropna()
    bp = ax.boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.6),
                    medianprops=dict(color='#C44E52', linewidth=2))
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Outlier Detection — Priority Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Check extreme values against clinical reference ranges
print('Clinical plausibility check:')
print(f'  Glucose max: {df["Glucose"].max()} mg/dL  — values >500 are unusual but possible in DKA')
print(f'  BMI max: {df["BMI"].max()} kg/m²  — extreme obesity, plausible')
print(f'  BP max: {df["BloodPressure"].max()} mmHg  — hypertensive crisis range, plausible')
print(f'  Insulin max: {df["Insulin"].max()} μU/mL  — very high, consider as outlier in modeling')

## 8. EDA Summary & Decisions for Preprocessing

| Finding | Impact | Decision |
|---|---|---|
| Zero-value missing data in 5 features | High — affects model inputs directly | Replace with NaN → median imputation by outcome group |
| Class imbalance (65/35) | Medium — accuracy unreliable | Use AUC-ROC as primary metric; stratified CV |
| Glucose: strongest predictor | Expected — diagnostic criterion | Keep as-is; monitor for data leakage |
| Insulin: 48.7% missing | High — feature may be unreliable | Impute but monitor feature importance |
| Outliers in Insulin, BMI | Low-medium | Cap at 99th percentile after imputation |
| No duplicate rows | — | Confirmed |
| No categorical features | — | No encoding needed |

**→ Proceed to notebook 02: Preprocessing & Feature Engineering**

In [ ]:
# Save cleaned version (zeros → NaN) for next notebook
df_clean.to_csv('../data/diabetes_nulls_flagged.csv', index=False)
print('Saved: data/diabetes_nulls_flagged.csv')
print(f'Shape: {df_clean.shape}')
print(f'NaN counts:\n{df_clean.isnull().sum()}')